In [ ]:
# ============================================================
# CÉLULA 1 — CARREGAMENTO DO AGENTE PLANAPP
# ============================================================
#%pip install "mcp"
import importlib
import agent_jupyter

agent_jupyter = importlib.reload(agent_jupyter)

PlanAppAgent = agent_jupyter.PlanAppAgent

In [ ]:
# ============================================================
# CÉLULA 2 — INTERFACE DO PLANAPP AI
# ============================================================

import asyncio
import html
import ipywidgets as widgets
from IPython.display import display


# ============================================================
# TÍTULO
# ============================================================

titulo = widgets.HTML(
    value="""
    <h2 style="margin-bottom:4px;">
        📡 PlanApp AI
    </h2>

    <div style="color:#666;">
        Agente de IA para planejamento e avaliação de enlaces de rádio
    </div>
    """
)


# ============================================================
# CAMPO DE ENTRADA
# ============================================================

entrada = widgets.Textarea(
    placeholder=(
        "Ex.: Analise um enlace entre a Praça da Sé e "
        "o Largo do Paissandu em São Paulo."
    ),
    layout=widgets.Layout(
        width="100%",
        height="100px",
    ),
)


# ============================================================
# BOTÕES
# ============================================================

botao = widgets.Button(
    description="🔍 Analisar enlace",
    button_style="primary",
    layout=widgets.Layout(
        width="180px"
    ),
)

botao_limpar = widgets.Button(
    description="↺ Nova análise",
    layout=widgets.Layout(
        width="150px"
    ),
)


# ============================================================
# STATUS PRINCIPAL
# ============================================================

status = widgets.HTML(
    value="""
    <div style="
        padding:10px;
        margin-top:10px;
        border-radius:6px;
        background:#e8f5e9;
        color:#2e7d32;
        font-weight:500;
    ">
        🟢 Pronto para analisar
    </div>
    """
)


# ============================================================
# LOG AO VIVO
# ============================================================

log_execucao = widgets.HTML(
    value="",
    layout=widgets.Layout(
        width="100%"
    )
)


# ============================================================
# RESULTADO
# ============================================================

resposta = widgets.HTML(
    value=""
)


# ============================================================
# EXEMPLOS
# ============================================================

exemplos = widgets.HTML(
    value="""
    <div style="
        margin-top:15px;
        color:#666;
    ">

        <b>Exemplos:</b><br>

        • Analise um enlace entre a Praça da Sé e o Largo do Paissandu.<br>

        • Avalie um enlace entre Curitiba e São José dos Pinhais,
          com antenas de 20 metros e frequência de 18 GHz.<br>

        • Analise o enlace entre dois endereços em Curitiba.

    </div>
    """
)


# ============================================================
# HISTÓRICO VISUAL
# ============================================================

historico_status = []


# ============================================================
# ATUALIZA STATUS
# ============================================================

def atualizar_status(
    texto,
    tipo="processing",
):

    global historico_status

    texto = str(texto)

    # --------------------------------------------------------
    # Cores do status principal
    # --------------------------------------------------------

    if tipo in ("ok", "success"):

        fundo = "#e8f5e9"
        cor = "#2e7d32"

    elif tipo in ("erro", "error"):

        fundo = "#ffebee"
        cor = "#c62828"

    else:

        fundo = "#fff8e1"
        cor = "#8a6d00"


    # --------------------------------------------------------
    # STATUS ATUAL
    # --------------------------------------------------------

    status.value = f"""
    <div style="
        padding:10px;
        margin-top:10px;
        border-radius:6px;
        background:{fundo};
        color:{cor};
        font-weight:500;
    ">
        {html.escape(texto)}
    </div>
    """


    # --------------------------------------------------------
    # ADICIONA AO LOG VISUAL
    # --------------------------------------------------------

    historico_status.append(
        texto
    )


    linhas = []

    for item in historico_status:

        # Pequeno destaque visual para cada linha
        if item.startswith("🔧"):
            estilo = "font-weight:600;"

        elif item.startswith("📥"):
            estilo = "margin-left:20px;"

        elif item.startswith("🟢"):
            estilo = "font-weight:600;"

        elif item.startswith("❌"):
            estilo = "font-weight:600;"

        else:
            estilo = ""

        linhas.append(
            f"""
            <div style="
                padding:3px 0;
                {estilo}
            ">
                {html.escape(item)}
            </div>
            """
        )


    log_execucao.value = f"""
    <div style="
        margin-top:10px;
        padding:12px;
        border:1px solid #ddd;
        border-radius:8px;
        background:#fafafa;
        font-family:monospace;
        font-size:13px;
        line-height:1.45;
        max-height:350px;
        overflow-y:auto;
    ">

        <div style="
            font-family:sans-serif;
            font-weight:bold;
            margin-bottom:8px;
        ">
            🔄 Execução do agente
        </div>

        {''.join(linhas)}

    </div>
    """


# ============================================================
# AGENTE
# ============================================================

agent = PlanAppAgent(
    progress_callback=atualizar_status
)


# ============================================================
# CONTROLE
# ============================================================

executando = False
task_atual = None


# ============================================================
# EXECUÇÃO ASSÍNCRONA
# ============================================================

async def executar_analise_async(texto):

    global executando

    try:

        # ----------------------------------------------------
        # LIMPA LOG VISUAL
        # ----------------------------------------------------

        historico_status.clear()

        log_execucao.value = ""

        resposta.value = ""


        # ----------------------------------------------------
        # CHAMADA DO AGENTE
        # ----------------------------------------------------

        resultado = await agent.ask(
            texto
        )


        # ----------------------------------------------------
        # RESULTADO FINAL
        # ----------------------------------------------------

        resposta.value = f"""
        <div style="
            margin-top:15px;
            padding:15px;
            border:1px solid #ddd;
            border-radius:8px;
            background:white;
        ">

            <div style="
                font-size:16px;
                font-weight:bold;
                margin-bottom:10px;
            ">
                📊 Resultado da análise
            </div>

            <div style="
                white-space:pre-wrap;
                line-height:1.5;
            ">
                {html.escape(resultado or "")}
            </div>

        </div>
        """


    except asyncio.CancelledError:

        atualizar_status(
            "⚠️ Análise cancelada.",
            "erro",
        )

        resposta.value = ""

        raise


    except Exception as e:

        atualizar_status(
            f"❌ Erro: {str(e)}",
            "erro",
        )

        resposta.value = ""


    finally:

        executando = False

        botao.disabled = False
        botao_limpar.disabled = False


# ============================================================
# CALLBACK DO BOTÃO
# ============================================================

def executar_analise(b):

    global executando
    global task_atual


    # --------------------------------------------------------
    # Evita duas análises simultâneas
    # --------------------------------------------------------

    if executando:

        return


    # --------------------------------------------------------
    # Texto
    # --------------------------------------------------------

    texto = entrada.value.strip()

    if not texto:

        atualizar_status(
            "⚠️ Digite uma solicitação antes de analisar.",
            "erro",
        )

        return


    # --------------------------------------------------------
    # Estado
    # --------------------------------------------------------

    executando = True

    botao.disabled = True
    botao_limpar.disabled = True

    resposta.value = ""

    historico_status.clear()
    log_execucao.value = ""


    # --------------------------------------------------------
    # PRIMEIRA MENSAGEM
    # --------------------------------------------------------

    atualizar_status(
        "🔵 Iniciando agente Qwen...",
        "processing",
    )


    # --------------------------------------------------------
    # CRIA TASK
    # --------------------------------------------------------

    task_atual = asyncio.create_task(
        executar_analise_async(
            texto
        )
    )


    # --------------------------------------------------------
    # CALLBACK DA TASK
    # --------------------------------------------------------

    def finalizar_task(task):

        global task_atual

        try:

            task.result()

        except asyncio.CancelledError:

            pass

        except Exception as e:

            print(
                ">>> ERRO NÃO CAPTURADO:",
                repr(e)
            )

        finally:

            task_atual = None


    task_atual.add_done_callback(
        finalizar_task
    )


# ============================================================
# NOVA ANÁLISE
# ============================================================

def nova_analise(b):

    global executando
    global task_atual


    if executando:

        atualizar_status(
            "⏳ Uma análise ainda está em execução.",
            "processing",
        )

        return


    entrada.value = ""

    resposta.value = ""

    historico_status.clear()

    log_execucao.value = ""


    atualizar_status(
        "🟢 Pronto para uma nova análise",
        "ok",
    )


    if agent is not None:

        agent.reset_history()


# ============================================================
# CALLBACKS
# ============================================================

botao.on_click(
    executar_analise
)

botao_limpar.on_click(
    nova_analise
)


# ============================================================
# MOSTRA INTERFACE
# ============================================================

display(
    titulo,
    entrada,
    widgets.HBox(
        [
            botao,
            botao_limpar,
        ]
    ),
    status,
    log_execucao,
    resposta,
    exemplos,
)